# Court Citation Concept Enrichment Smoke Test

This notebook reads `court_consideration.csv`, enriches **10 court citation rows** with query-neutral legal descriptors, and writes both JSONL and CSV preview outputs.

The enrichment intentionally avoids synthetic user questions. It extracts:
- legal area / topic / subtopic / micro-topic
- English legal concepts
- original-language legal terms
- statute and case anchors
- doctrinal rule / legal test
- fact-pattern tags
- procedural context
- paragraph role / authority role / outcome signal
- retrieval views derived from those descriptors

Use this as a qualitative smoke test before scaling to the full corpus.


In [ ]:
# Optional Kaggle install cell
# Run only if vLLM / transformers are not already available in your environment.
# On Kaggle this may take time.

# !pip install -q -U "vllm>=0.6.0" "transformers>=4.45.0" accelerate pandas tqdm


In [ ]:
from pathlib import Path
import os
import re
import json
import time
import random
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional

import pandas as pd
from tqdm.auto import tqdm


@dataclass
class CONFIG:
    # Update this path if your CSV is in a Kaggle dataset directory.
    input_csv: str = "/kaggle/input/your-dataset/court_consideration.csv"

    # Fallback: if the CSV is uploaded into the notebook working directory.
    fallback_input_csv: str = "court_consideration.csv"

    output_dir: str = "/kaggle/working"
    output_jsonl: str = "enriched_court_citations_10.jsonl"
    output_preview_csv: str = "enriched_court_citations_10_preview.csv"

    # Number of rows for smoke test.
    n_rows: int = 10

    # Optional deterministic row selection.
    # Set to None for first 10 valid rows.
    random_seed: Optional[int] = None

    # If True, sample random valid rows instead of taking the first 10.
    sample_random: bool = False

    # Input text budget per citation. Keep enough context for legal meaning.
    text_chars: int = 5500

    # Model settings.
    model_name: str = "Qwen/Qwen3-8B-AWQ"
    tensor_parallel_size: int = 1
    gpu_memory_utilization: float = 0.70
    max_model_len: int = 4096
    enforce_eager: bool = True

    # Generation settings.
    batch_size: int = 10
    max_new_tokens: int = 768
    temperature: float = 0.0
    top_p: float = 1.0

    # Qwen3 thinking mode should be disabled for JSON indexing.
    enable_thinking: bool = False

    # Set this if Kaggle has FlashInfer/JIT issues.
    attention_backend: str = "TRITON_ATTN"


cfg = CONFIG()

# Useful Kaggle/vLLM environment defaults.
os.environ.setdefault("VLLM_ATTENTION_BACKEND", cfg.attention_backend)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

Path(cfg.output_dir).mkdir(parents=True, exist_ok=True)

print(asdict(cfg))


In [ ]:
def resolve_input_path(cfg: CONFIG) -> Path:
    candidates = [
        Path(cfg.input_csv),
        Path(cfg.fallback_input_csv),
        Path("/kaggle/working") / cfg.fallback_input_csv,
        Path("/mnt/data") / cfg.fallback_input_csv,
    ]
    for p in candidates:
        if p.exists():
            return p

    # Helpful search for Kaggle.
    possible = list(Path("/kaggle/input").glob("**/court_consideration.csv")) if Path("/kaggle/input").exists() else []
    if possible:
        return possible[0]

    raise FileNotFoundError(
        "Could not find court_consideration.csv. "
        "Update CONFIG.input_csv to the correct file path."
    )


input_path = resolve_input_path(cfg)
print("Using input:", input_path)

df = pd.read_csv(input_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head(3)


In [ ]:
def pick_column(columns: List[str], preferred: List[str], contains_any: List[str]) -> Optional[str]:
    lower_map = {c.lower(): c for c in columns}
    for name in preferred:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    for c in columns:
        lc = c.lower()
        if any(token in lc for token in contains_any):
            return c
    return None


columns = list(df.columns)

citation_col = pick_column(
    columns,
    preferred=["citation", "cite", "court_citation", "authority_citation", "consideration_citation"],
    contains_any=["citation", "cite", "bge"]
)

text_col = pick_column(
    columns,
    preferred=["text", "consideration_text", "paragraph_text", "content", "raw_text", "body"],
    contains_any=["text", "content", "paragraph", "consideration", "body"]
)

if citation_col is None:
    raise ValueError(
        "Could not infer citation column. "
        "Set citation_col manually to the citation column name."
    )

if text_col is None:
    raise ValueError(
        "Could not infer text column. "
        "Set text_col manually to the text/content column name."
    )

print("citation_col:", citation_col)
print("text_col:", text_col)

valid = df[df[citation_col].notna() & df[text_col].notna()].copy()
valid[text_col] = valid[text_col].astype(str)
valid = valid[valid[text_col].str.strip().str.len() > 50].copy()

if cfg.sample_random:
    valid_10 = valid.sample(n=min(cfg.n_rows, len(valid)), random_state=cfg.random_seed)
else:
    valid_10 = valid.head(cfg.n_rows)

valid_10 = valid_10.reset_index(drop=False).rename(columns={"index": "_source_row"})
print("Selected rows:", len(valid_10))
valid_10[[citation_col, text_col]].head(10)


In [ ]:
ENRICHMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "legal_area": {"type": "string"},
        "legal_domain_path": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 2,
            "maxItems": 6
        },
        "topic": {"type": "string"},
        "subtopic": {"type": "string"},
        "micro_topic": {"type": "string"},
        "concepts_en": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 4,
            "maxItems": 14
        },
        "terms_original": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 4,
            "maxItems": 18
        },
        "statute_anchors": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 10
        },
        "case_anchors": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 10
        },
        "doctrinal_rule": {"type": "string"},
        "legal_test": {"type": "string"},
        "fact_pattern_tags": {
            "type": "array",
            "items": {"type": "string"},
            "maxItems": 10
        },
        "procedural_context": {"type": "string"},
        "paragraph_role": {
            "type": "string",
            "enum": [
                "holding",
                "reasoning",
                "facts",
                "procedural_history",
                "citation",
                "dissent",
                "neutral"
            ]
        },
        "authority_role": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": [
                    "leading_decision",
                    "legal_test",
                    "constitutional_standard",
                    "statutory_interpretation",
                    "standard_of_review",
                    "application_of_rule",
                    "distinguishing_case",
                    "background",
                    "procedural_rule",
                    "evidentiary_standard",
                    "official_intervention_limit",
                    "voting_rights_jurisprudence",
                    "neutral"
                ]
            },
            "maxItems": 5
        },
        "outcome_signal": {
            "type": "string",
            "enum": [
                "granted",
                "dismissed",
                "remanded",
                "partially_granted",
                "inadmissible",
                "neutral"
            ]
        },
        "specificity_score": {
            "type": "number",
            "minimum": 0,
            "maximum": 1
        }
    },
    "required": [
        "legal_area",
        "legal_domain_path",
        "topic",
        "subtopic",
        "micro_topic",
        "concepts_en",
        "terms_original",
        "statute_anchors",
        "case_anchors",
        "doctrinal_rule",
        "legal_test",
        "fact_pattern_tags",
        "procedural_context",
        "paragraph_role",
        "authority_role",
        "outcome_signal",
        "specificity_score"
    ],
    "additionalProperties": False
}


SYSTEM_PROMPT = '''You are a Swiss legal indexing assistant.

Create compact retrieval metadata for one Swiss court citation.

Important rules:
- Do NOT generate user questions.
- Do NOT invent facts beyond the provided text.
- Extract grounded legal descriptors only.
- Prefer precise, citation-specific legal concepts over generic words.
- Keep descriptors compact and useful for retrieval.
- Use English for legal_area, topic, subtopic, micro_topic, concepts_en, doctrinal_rule, legal_test, fact_pattern_tags, procedural_context.
- Preserve important German/French/Italian legal terms in terms_original.
- Include statute and case anchors only if present or clearly referenced in the text.
- If outcome is not clear from this passage, use "neutral".
- Return valid JSON only, matching the schema.
'''


def make_user_prompt(citation: str, text: str) -> str:
    text = str(text).strip()
    if len(text) > cfg.text_chars:
        text = text[:cfg.text_chars] + "\n...[TRUNCATED]"
    return f'''Citation: {citation}

Text:
{text}

Return the enrichment JSON only.'''


In [ ]:
def extract_json_object(s: str) -> Dict[str, Any]:
    if not isinstance(s, str):
        raise ValueError("Model output is not a string.")

    s = s.strip()

    # Remove common markdown fences if present.
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    # Remove Qwen thinking tags if a model emits them despite disabled thinking.
    s = re.sub(r"<think>.*?</think>", "", s, flags=re.DOTALL | re.IGNORECASE).strip()

    try:
        return json.loads(s)
    except Exception:
        pass

    start = s.find("{")
    end = s.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError(f"No JSON object found in output: {s[:500]}")

    return json.loads(s[start:end + 1])


def normalize_list(value: Any, max_items: int) -> List[str]:
    if value is None:
        return []
    if isinstance(value, str):
        value = [value]
    if not isinstance(value, list):
        return []
    cleaned = []
    seen = set()
    for item in value:
        item = str(item).strip()
        if not item:
            continue
        key = item.lower()
        if key not in seen:
            seen.add(key)
            cleaned.append(item)
        if len(cleaned) >= max_items:
            break
    return cleaned


def normalize_enrichment(obj: Dict[str, Any]) -> Dict[str, Any]:
    out = dict(obj)

    string_fields = [
        "legal_area",
        "topic",
        "subtopic",
        "micro_topic",
        "doctrinal_rule",
        "legal_test",
        "procedural_context",
        "paragraph_role",
        "outcome_signal",
    ]
    for field in string_fields:
        out[field] = str(out.get(field, "")).strip()

    out["legal_domain_path"] = normalize_list(out.get("legal_domain_path"), 6)
    out["concepts_en"] = normalize_list(out.get("concepts_en"), 14)
    out["terms_original"] = normalize_list(out.get("terms_original"), 18)
    out["statute_anchors"] = normalize_list(out.get("statute_anchors"), 10)
    out["case_anchors"] = normalize_list(out.get("case_anchors"), 10)
    out["fact_pattern_tags"] = normalize_list(out.get("fact_pattern_tags"), 10)
    out["authority_role"] = normalize_list(out.get("authority_role"), 5)

    valid_paragraph_roles = {
        "holding", "reasoning", "facts", "procedural_history", "citation", "dissent", "neutral"
    }
    if out["paragraph_role"] not in valid_paragraph_roles:
        out["paragraph_role"] = "neutral"

    valid_outcomes = {
        "granted", "dismissed", "remanded", "partially_granted", "inadmissible", "neutral"
    }
    if out["outcome_signal"] not in valid_outcomes:
        out["outcome_signal"] = "neutral"

    try:
        score = float(out.get("specificity_score", 0.0))
    except Exception:
        score = 0.0
    out["specificity_score"] = max(0.0, min(1.0, score))

    return out


def build_retrieval_views(citation: str, enrichment: Dict[str, Any], raw_text: str) -> Dict[str, str]:
    def join(items):
        return " ".join([str(x).strip() for x in items if str(x).strip()])

    return {
        "semantic_concepts_en": join([
            enrichment.get("legal_area", ""),
            enrichment.get("topic", ""),
            enrichment.get("subtopic", ""),
            enrichment.get("micro_topic", ""),
            *enrichment.get("concepts_en", []),
        ]),
        "topic_path": " > ".join(enrichment.get("legal_domain_path", [])),
        "original_terms_view": join(enrichment.get("terms_original", [])),
        "statute_anchor_view": join([
            *enrichment.get("statute_anchors", []),
            enrichment.get("legal_area", ""),
            enrichment.get("topic", ""),
            enrichment.get("subtopic", ""),
        ]),
        "case_anchor_view": join([citation, *enrichment.get("case_anchors", [])]),
        "legal_rule_view": join([
            enrichment.get("doctrinal_rule", ""),
            enrichment.get("legal_test", ""),
        ]),
        "fact_pattern_view": join([
            enrichment.get("procedural_context", ""),
            *enrichment.get("fact_pattern_tags", []),
        ]),
        "raw_context": str(raw_text)[:cfg.text_chars],
    }


def validate_enrichment(e: Dict[str, Any]) -> List[str]:
    errors = []
    for field in ENRICHMENT_SCHEMA["required"]:
        if field not in e:
            errors.append(f"missing:{field}")

    if len(e.get("concepts_en", [])) < 3:
        errors.append("weak:concepts_en")
    if len(e.get("terms_original", [])) < 2:
        errors.append("weak:terms_original")
    if not e.get("topic") or not e.get("subtopic"):
        errors.append("weak:topic")
    if not e.get("doctrinal_rule") and not e.get("legal_test"):
        errors.append("weak:rule_or_test")

    generic_bad = {"law", "court", "case", "decision", "legal", "rights", "appeal"}
    concept_lower = {c.lower().strip() for c in e.get("concepts_en", [])}
    if concept_lower and len(concept_lower - generic_bad) < 3:
        errors.append("too_generic:concepts_en")

    return errors


In [ ]:
# Load vLLM model.
# If this cell fails on Kaggle, first run the optional install cell and restart the notebook.

from vllm import LLM, SamplingParams

try:
    from vllm.sampling_params import GuidedDecodingParams
except Exception:
    GuidedDecodingParams = None

print("Loading model:", cfg.model_name)

llm_kwargs = dict(
    model=cfg.model_name,
    tensor_parallel_size=cfg.tensor_parallel_size,
    gpu_memory_utilization=cfg.gpu_memory_utilization,
    max_model_len=cfg.max_model_len,
    enforce_eager=cfg.enforce_eager,
    trust_remote_code=True,
    disable_custom_all_reduce=True,
)

llm = LLM(**llm_kwargs)
print("Model loaded.")


In [ ]:
def build_sampling_params() -> SamplingParams:
    kwargs = dict(
        temperature=cfg.temperature,
        top_p=cfg.top_p,
        max_tokens=cfg.max_new_tokens,
    )

    # Try guided JSON if this vLLM version supports it.
    if GuidedDecodingParams is not None:
        try:
            kwargs["guided_decoding"] = GuidedDecodingParams(json=ENRICHMENT_SCHEMA)
            print("Using vLLM guided JSON decoding.")
        except Exception as exc:
            print("Guided JSON unavailable, using prompt-only JSON control:", repr(exc))

    return SamplingParams(**kwargs)


def build_messages(row: pd.Series) -> List[Dict[str, str]]:
    citation = str(row[citation_col])
    text = str(row[text_col])

    user_prompt = make_user_prompt(citation, text)

    # Qwen chat template understands enable_thinking in recent transformers.
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]


def render_prompt(tokenizer, messages: List[Dict[str, str]]) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=cfg.enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


sampling_params = build_sampling_params()
tokenizer = llm.get_tokenizer()

prompts = [
    render_prompt(tokenizer, build_messages(row))
    for _, row in valid_10.iterrows()
]

print("Built prompts:", len(prompts))
print(prompts[0][:1000])


In [ ]:
# Run enrichment for 10 rows.

start = time.time()
outputs = llm.generate(prompts, sampling_params)
elapsed = time.time() - start

records = []
parse_failures = []

for i, (row_idx, row) in enumerate(valid_10.iterrows()):
    citation = str(row[citation_col])
    raw_text = str(row[text_col])
    raw_output = outputs[i].outputs[0].text if outputs[i].outputs else ""

    try:
        enrichment = extract_json_object(raw_output)
        enrichment = normalize_enrichment(enrichment)
        errors = validate_enrichment(enrichment)
        retrieval_views = build_retrieval_views(citation, enrichment, raw_text)

        record = {
            "citation": citation,
            "source_row": int(row["_source_row"]),
            "source_family": "court",
            "rag_enrichment": enrichment,
            "retrieval_views": retrieval_views,
            "enrichment_quality": {
                "method": "qwen3_8b_awq_concept_descriptor_enrichment",
                "question_generation_used": False,
                "grounded_references_only": True,
                "json_valid": True,
                "validation_errors": errors,
                "low_value_paragraph": False if not errors else ("too_generic:concepts_en" in errors),
            },
            "_raw_model_output": raw_output,
        }
    except Exception as exc:
        parse_failures.append((citation, repr(exc), raw_output[:1000]))
        record = {
            "citation": citation,
            "source_row": int(row["_source_row"]),
            "source_family": "court",
            "rag_enrichment": None,
            "retrieval_views": {
                "raw_context": raw_text[:cfg.text_chars],
                "case_anchor_view": citation,
            },
            "enrichment_quality": {
                "method": "qwen3_8b_awq_concept_descriptor_enrichment",
                "question_generation_used": False,
                "json_valid": False,
                "validation_errors": [f"parse_error:{repr(exc)}"],
                "low_value_paragraph": True,
            },
            "_raw_model_output": raw_output,
        }

    records.append(record)

print(f"Generated {len(records)} records in {elapsed:.2f}s")
print(f"Parse failures: {len(parse_failures)}")

if parse_failures:
    print(parse_failures[0][0])
    print(parse_failures[0][1])
    print(parse_failures[0][2][:1000])


In [ ]:
# Write JSONL and preview CSV.

jsonl_path = Path(cfg.output_dir) / cfg.output_jsonl
preview_path = Path(cfg.output_dir) / cfg.output_preview_csv

with jsonl_path.open("w", encoding="utf-8") as f:
    for rec in records:
        # Drop raw model output from JSONL if you do not want debug payloads.
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

preview_rows = []
for rec in records:
    e = rec.get("rag_enrichment") or {}
    rv = rec.get("retrieval_views") or {}
    q = rec.get("enrichment_quality") or {}
    preview_rows.append({
        "citation": rec.get("citation"),
        "source_row": rec.get("source_row"),
        "legal_area": e.get("legal_area"),
        "topic": e.get("topic"),
        "subtopic": e.get("subtopic"),
        "micro_topic": e.get("micro_topic"),
        "concepts_en": " | ".join(e.get("concepts_en", [])),
        "terms_original": " | ".join(e.get("terms_original", [])),
        "statute_anchors": " | ".join(e.get("statute_anchors", [])),
        "case_anchors": " | ".join(e.get("case_anchors", [])),
        "doctrinal_rule": e.get("doctrinal_rule"),
        "legal_test": e.get("legal_test"),
        "fact_pattern_tags": " | ".join(e.get("fact_pattern_tags", [])),
        "procedural_context": e.get("procedural_context"),
        "paragraph_role": e.get("paragraph_role"),
        "authority_role": " | ".join(e.get("authority_role", [])),
        "outcome_signal": e.get("outcome_signal"),
        "specificity_score": e.get("specificity_score"),
        "validation_errors": " | ".join(q.get("validation_errors", [])),
        "semantic_concepts_en": rv.get("semantic_concepts_en"),
        "topic_path": rv.get("topic_path"),
        "original_terms_view": rv.get("original_terms_view"),
    })

preview_df = pd.DataFrame(preview_rows)
preview_df.to_csv(preview_path, index=False)

print("Wrote:", jsonl_path)
print("Wrote:", preview_path)

preview_df


In [ ]:
# Qualitative inspection helper:
# Look for overly generic enrichment. Good rows should have narrow micro_topic,
# specific original terms, and specific concept descriptors.

pd.set_option("display.max_colwidth", 300)

cols = [
    "citation",
    "topic",
    "subtopic",
    "micro_topic",
    "concepts_en",
    "terms_original",
    "doctrinal_rule",
    "validation_errors",
]

preview_df[cols]


## What to check manually

For each of the 10 rows, check:

1. **Specificity**: does `micro_topic` distinguish the citation from generic citations?
2. **Concept quality**: are `concepts_en` precise legal concepts rather than generic words?
3. **Original terms**: did it preserve important German/French/Italian terms?
4. **Anchors**: did it capture statutes and cited cases without hallucinating?
5. **Retrieval views**: would these fields make the citation surface for semantic search?
6. **No synthetic questions**: `question_generation_used` should remain `False`.

If the output is too verbose, lower `CONFIG.max_new_tokens`.
If it misses important legal terms, increase `CONFIG.text_chars` or improve the prompt examples.
